In [0]:
from pyspark.sql import functions as F


def build_silver_weather(weather_bronze):
    weather_arrays = weather_bronze.select(
        "*",
        F.col("hourly.time").alias("_time"),
        F.col("hourly.temperature_2m").alias("_temperature"),
        F.col("hourly.precipitation").alias("_precipitation"),
        F.col("hourly.wind_speed_10m").alias("_wind_speed"),
    )

    array_checks = weather_arrays.select(
        "_source_file",
        F.size("_time").alias("time_count"),
        F.size("_temperature").alias("temperature_count"),
        F.size("_precipitation").alias("precipitation_count"),
        F.size("_wind_speed").alias("wind_speed_count"),
    )

    bad_arrays = array_checks.filter(
        (F.col("time_count") != F.col("temperature_count"))
        | (F.col("time_count") != F.col("precipitation_count"))
        | (F.col("time_count") != F.col("wind_speed_count"))
    )

    if bad_arrays.limit(1).count() > 0:
        raise ValueError(
            "Weather hourly arrays are misaligned."
        )

    weather_zipped = weather_arrays.withColumn(
        "_hourly_zipped",
        F.arrays_zip(
            "_time",
            "_temperature",
            "_precipitation",
            "_wind_speed",
        ),
    )

    weather_exploded = weather_zipped.select(
        "*",
        F.posexplode("_hourly_zipped").alias(
            "hour_index",
            "weather_hour",
        ),
    )

    silver_weather = weather_exploded.select(
        F.to_timestamp(
            F.col("weather_hour._time"),
            "yyyy-MM-dd'T'HH:mm",
        ).alias("weather_hour_local"),

        F.to_date(
            F.col("weather_hour._time")
        ).alias("weather_date_local"),

        F.col("weather_hour._temperature")
        .cast("double")
        .alias("temperature_2m_c"),

        F.col("weather_hour._precipitation")
        .cast("double")
        .alias("precipitation_mm"),

        F.col("weather_hour._wind_speed")
        .cast("double")
        .alias("wind_speed_10m_kmh"),

        F.col("timezone").alias("timezone"),

        F.col("_source_file").alias("source_file"),

        F.col("_ingested_at").alias("ingested_at"),
    )

    return silver_weather

In [0]:
BRONZE_WEATHER_TABLE = (
    "nyc_mobility.nyc_bronze.bronze_weather_raw"
)

SILVER_WEATHER_TABLE = (
    "nyc_mobility.nyc_silver.silver_weather_hourly"
)

In [0]:
weather_bronze = spark.table(
    BRONZE_WEATHER_TABLE
)

print("Bronze rows:", weather_bronze.count())

In [0]:
silver_weather = build_silver_weather(
    weather_bronze
)

In [0]:
silver_weather.printSchema()

display(
    silver_weather
    .orderBy("weather_hour_local")
    .limit(20)
)

In [0]:
duplicate_hours = (
    silver_weather
    .groupBy("weather_hour_local")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_count = duplicate_hours.count()

print("Duplicate weather hours:", duplicate_count)

if duplicate_count > 0:
    display(duplicate_hours)
    raise ValueError(
        "Duplicate weather hours detected."
    )

In [0]:
required_nulls = silver_weather.agg(
    F.sum(
        F.col("weather_hour_local")
        .isNull()
        .cast("int")
    ).alias("weather_hour_nulls"),

    F.sum(
        F.col("timezone")
        .isNull()
        .cast("int")
    ).alias("timezone_nulls"),

    F.sum(
        F.col("source_file")
        .isNull()
        .cast("int")
    ).alias("source_file_nulls"),

    F.sum(
        F.col("ingested_at")
        .isNull()
        .cast("int")
    ).alias("ingested_at_nulls"),
)

display(required_nulls)

In [0]:
coverage = silver_weather.agg(
    F.min("weather_hour_local").alias("first_hour"),
    F.max("weather_hour_local").alias("last_hour"),
    F.count("*").alias("row_count"),
)

display(coverage)

In [0]:
monthly_counts = (
    silver_weather
    .withColumn(
        "month",
        F.date_format(
            "weather_hour_local",
            "yyyy-MM",
        ),
    )
    .groupBy("month")
    .count()
    .orderBy("month")
)

display(monthly_counts)

In [0]:
(
    silver_weather.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        SILVER_WEATHER_TABLE
    )
)